# Traffic Signal OpenEnv: Final Submission Training Notebook

This is the canonical self-contained training notebook used for the final submission flow. It does not clone the repository during training.

The stable final path uses a small 1B instruct model with PEFT LoRA, schema SFT warmup, a schema gate, and a manual GRPO-style policy optimization loop against the live OpenEnv traffic Space. It includes graceful retries for the environment API, optional W&B tracking, and optional Hugging Face Space artifact upload.

Required environment variables:

- `ENV_URL`: live traffic environment URL, for example `https://guuru-dev-traffic-signal-openenv-2.hf.space`
- `HF_TOKEN`: optional but required for uploading artifacts
- `WANDB_API_KEY`: optional but recommended for experiment tracking
- `WANDB_PROJECT`: optional, defaults to `traffic-signal-openenv`
- `WANDB_RUN_NAME`: optional, auto-generated when missing

In [ ]:
# Cell 1: Install stable training stack
# Run once, then restart the kernel before continuing.

import sys
import subprocess


def run(cmd):
    print("\n$", cmd)
    subprocess.check_call(cmd, shell=True)

run(f"{sys.executable} -m pip install --upgrade pip setuptools wheel")

# Keep the final submission path independent of Unsloth/TRL runtime quirks.
run(f"{sys.executable} -m pip uninstall -y trl unsloth unsloth-zoo mergekit llm-blender || true")

run(
    f"{sys.executable} -m pip install --upgrade "
    "transformers peft accelerate bitsandbytes datasets "
    "huggingface_hub wandb requests matplotlib pandas numpy scipy "
    "sentencepiece protobuf einops packaging safetensors"
)

print("Install complete. Restart kernel, then run from Cell 2.")

In [ ]:
# Cell 2: Configure environment, auth, W&B, and health checks

import os
import sys
import gc
import re
import csv
import json
import time
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
import wandb
from huggingface_hub import HfApi, login

SEED = int(os.getenv("SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ENV_URL = os.getenv("ENV_URL", "https://guuru-dev-traffic-signal-openenv-2.hf.space").rstrip("/")
SPACE_REPO = os.getenv("SPACE_REPO", "Guuru-DEV/traffic-signal-openenv-2")
MODEL_NAME = os.getenv("MODEL_NAME", "unsloth/Llama-3.2-1B-Instruct")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "traffic-signal-openenv")
WANDB_RUN_NAME = os.getenv("WANDB_RUN_NAME", f"openenv-final-1b-peft-{int(time.time())}")
HF_TOKEN = os.getenv("HF_TOKEN")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")

Path("outputs").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)
Path("plots").mkdir(exist_ok=True)

print("Python:", sys.version)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("ENV_URL:", ENV_URL)
print("SPACE_REPO:", SPACE_REPO)
print("MODEL_NAME:", MODEL_NAME)
print("WANDB_RUN_NAME:", WANDB_RUN_NAME)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected. Enable a GPU runtime before training.")

api = None
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    api = HfApi(token=HF_TOKEN)
    print("HF auth OK")
else:
    print("HF_TOKEN missing; artifact upload will be skipped.")

use_wandb = False
if WANDB_API_KEY:
    try:
        wandb.login(key=WANDB_API_KEY, relogin=True)
        wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config={
                "model_name": MODEL_NAME,
                "env_url": ENV_URL,
                "space_repo": SPACE_REPO,
                "seed": SEED,
                "pipeline": "peft_lora_manual_sft_manual_grpo",
            },
        )
        use_wandb = True
        print("W&B initialized:", wandb.run.url)
    except Exception as exc:
        print("W&B init failed; continuing without W&B:", repr(exc))
        wandb = None
else:
    print("WANDB_API_KEY missing; W&B disabled.")


def health_check(retries=8, timeout=60):
    last = None
    for attempt in range(retries):
        try:
            t0 = time.time()
            response = requests.get(f"{ENV_URL}/health", timeout=timeout)
            response.raise_for_status()
            print("Space health:", response.json())
            print("health_latency_sec:", round(time.time() - t0, 2))
            return response.json()
        except Exception as exc:
            last = exc
            wait = min(30, 3 * (attempt + 1))
            print(f"Health check failed ({attempt + 1}/{retries}): {exc!r}; retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"Environment health check failed: {last!r}")

health_check()

In [ ]:
# Cell 3: Load 1B base model with PEFT LoRA

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map="auto",
    attn_implementation="sdpa",
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.generation_config.do_sample = True
model.generation_config.temperature = 0.7
model.generation_config.top_p = 0.9
model.generation_config.pad_token_id = tokenizer.eos_token_id
model.generation_config.max_length = None

torch.cuda.empty_cache()
gc.collect()
print("Model ready. GPU memory allocated GB:", round(torch.cuda.memory_allocated() / 1e9, 2))

In [ ]:
# Cell 4: Manual SFT schema warmup

import itertools
from torch.utils.data import DataLoader
from torch.optim import AdamW

VALID_ACTIONS = {"KEEP", "SWITCH", "PHASE_0", "PHASE_1", "PHASE_2", "PHASE_3"}
CENTRAL_POLICY_KEYS = {"switch_penalty", "queue_urgency_weight", "emergency_boost", "corridor_priority", "balance_penalty"}

SCHEMA_PROMPT = (
    'Return exactly one compact JSON object with this schema: '
    '{"local_actions":{"NW":"ACTION","NE":"ACTION","SW":"ACTION","SE":"ACTION"},'
    '"central_action":{"queue_urgency_weight":0.0,"corridor_priority":0.0}}. '
    'ACTION must be one of KEEP, SWITCH, PHASE_0, PHASE_1, PHASE_2, PHASE_3. '
    'central_action should usually include 1-2 useful small numeric deltas. '
    'Allowed central_action keys: switch_penalty, queue_urgency_weight, emergency_boost, corridor_priority, balance_penalty. '
    'central_action values must be from -0.5 to 0.8. No prose, no markdown.'
)

schema_examples = [
    '{"local_actions":{"NW":"SWITCH","NE":"KEEP","SW":"PHASE_0","SE":"PHASE_1"},"central_action":{"queue_urgency_weight":0.4,"corridor_priority":0.3}}',
    '{"local_actions":{"NW":"PHASE_2","NE":"SWITCH","SW":"KEEP","SE":"PHASE_0"},"central_action":{"balance_penalty":0.3,"switch_penalty":0.2}}',
    '{"local_actions":{"NW":"PHASE_1","NE":"PHASE_3","SW":"SWITCH","SE":"KEEP"},"central_action":{"emergency_boost":0.5,"queue_urgency_weight":0.2}}',
    '{"local_actions":{"NW":"PHASE_0","NE":"PHASE_1","SW":"PHASE_2","SE":"SWITCH"},"central_action":{"corridor_priority":0.5,"balance_penalty":0.2}}',
    '{"local_actions":{"NW":"SWITCH","NE":"PHASE_0","SW":"PHASE_1","SE":"PHASE_2"},"central_action":{"switch_penalty":0.3,"queue_urgency_weight":0.3}}',
    '{"local_actions":{"NW":"PHASE_3","NE":"PHASE_2","SW":"SWITCH","SE":"PHASE_0"},"central_action":{"emergency_boost":0.4,"balance_penalty":0.3}}',
    '{"local_actions":{"NW":"PHASE_1","NE":"SWITCH","SW":"PHASE_3","SE":"KEEP"},"central_action":{"corridor_priority":0.2,"queue_urgency_weight":0.5}}',
    '{"local_actions":{"NW":"KEEP","NE":"PHASE_0","SW":"SWITCH","SE":"PHASE_2"},"central_action":{"balance_penalty":0.4,"emergency_boost":0.2}}',
]

texts = []
for _ in range(int(os.getenv("SFT_REPEAT", "140"))):
    random.shuffle(schema_examples)
    for example in schema_examples:
        texts.append(tokenizer.apply_chat_template(
            [
                {"role": "system", "content": "You output only compact valid JSON matching the required traffic schema."},
                {"role": "user", "content": SCHEMA_PROMPT},
                {"role": "assistant", "content": example},
            ],
            tokenize=False,
            add_generation_prompt=False,
        ))


def encode_text(text):
    enc = tokenizer(text, max_length=512, truncation=True, padding=False, return_tensors=None)
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}

encoded = [encode_text(t) for t in texts]


def collate(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        ids = x["input_ids"]
        mask = x["attention_mask"]
        pad_len = max_len - len(ids)
        input_ids.append(ids + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append(mask + [0] * pad_len)
        labels.append(ids + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long, device=model.device),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long, device=model.device),
        "labels": torch.tensor(labels, dtype=torch.long, device=model.device),
    }

loader = DataLoader(encoded, batch_size=int(os.getenv("SFT_BATCH_SIZE", "8")), shuffle=True, collate_fn=collate)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=float(os.getenv("SFT_LR", "3e-5")))
steps = int(os.getenv("SFT_STEPS", "100"))
grad_accum = int(os.getenv("SFT_GRAD_ACCUM", "2"))
running = iter(itertools.cycle(loader))

model.train()
optimizer.zero_grad(set_to_none=True)
print("Starting SFT schema warmup...")
for step in range(1, steps + 1):
    total_loss = 0.0
    for _ in range(grad_accum):
        batch = next(running)
        with torch.autocast("cuda", dtype=DTYPE):
            out = model(**batch)
            loss = out.loss / grad_accum
        loss.backward()
        total_loss += float(loss.detach().cpu()) * grad_accum
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    if step == 1 or step % 5 == 0:
        print(step, round(total_loss, 6))
        if use_wandb:
            wandb.log({"sft/loss": total_loss, "sft/step": step})

print("SFT schema warmup complete.")
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Cell 5: Schema parser and validation gate

LOCAL_KEY_ALIASES = {"localactions", "localaction", "localact", "local", "actions", "action"}
CENTRAL_KEY_ALIASES = {"centralaction", "centralactions", "central", "policy"}


def _alias_key(key):
    return re.sub(r"[^a-z0-9]", "", str(key).lower())


def normalize_action_value(value):
    s = str(value).upper().strip()
    s = re.sub(r"[^A-Z0-9_]", "", s)
    if s in VALID_ACTIONS:
        return s
    aliases = {"PHASE0": "PHASE_0", "P0": "PHASE_0", "PHASE1": "PHASE_1", "P1": "PHASE_1", "PHASE2": "PHASE_2", "P2": "PHASE_2", "PHASE3": "PHASE_3", "P3": "PHASE_3"}
    if s in aliases:
        return aliases[s]
    if "SWITCH" in s:
        return "SWITCH"
    if "KEEP" in s:
        return "KEEP"
    return None


def normalize_action_obj(obj):
    if not isinstance(obj, dict):
        return None
    local = None
    central = {}
    for key, value in obj.items():
        alias = _alias_key(key)
        if alias in LOCAL_KEY_ALIASES:
            local = value
        elif alias in CENTRAL_KEY_ALIASES:
            central = value
    if not isinstance(local, dict):
        return None

    clean_local = {}
    for direction in ("NW", "NE", "SW", "SE"):
        raw_value = None
        for key, value in local.items():
            if str(key).upper().strip() == direction:
                raw_value = value
                break
        if raw_value is None:
            return None
        action = normalize_action_value(raw_value)
        if action is None:
            return None
        clean_local[direction] = action

    clean_central = {}
    if isinstance(central, dict):
        for key, value in central.items():
            if key not in CENTRAL_POLICY_KEYS:
                continue
            try:
                clean_central[key] = max(-0.5, min(0.8, float(value)))
            except Exception:
                continue
    return {"local_actions": clean_local, "central_action": clean_central}


def extract_schema_json(text):
    decoder = json.JSONDecoder()
    best = None
    for idx, char in enumerate(str(text)):
        if char != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(str(text)[idx:])
        except Exception:
            continue
        normalized = normalize_action_obj(obj)
        if normalized is not None:
            best = normalized
    return best

valid_count = 0
central_count = 0
model.eval()
for i in range(8):
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": "You output only compact valid JSON matching the required traffic schema."},
            {"role": "user", "content": SCHEMA_PROMPT},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=96, do_sample=True, temperature=0.25, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    obj = extract_schema_json(text)
    ok = obj is not None
    has_central = bool(obj and obj.get("central_action"))
    valid_count += int(ok)
    central_count += int(has_central)
    print(f"SAMPLE {i + 1} valid={ok} has_central={has_central} extracted={obj}")

SCHEMA_READY = valid_count >= 7 and central_count >= 6
print(f"Schema validation: valid={valid_count}/8 central={central_count}/8")
print("SCHEMA_READY:", SCHEMA_READY)
if not SCHEMA_READY:
    raise RuntimeError("Schema warmup insufficient. Do not start policy optimization.")

In [ ]:
# Cell 6: Environment reward function with graceful retries

if not globals().get("SCHEMA_READY", False):
    raise RuntimeError("SCHEMA_READY is false. Run schema validation first.")

PROMPT_BANK_RAW = [
    "Hard multi-intersection control: choose local phase actions and small central deltas for queue pressure and corridor flow.",
    "Prevent spillback using valid local actions plus bounded central_action deltas. JSON only.",
    "Emergency-aware traffic control: include emergency_boost only if useful, keep deltas small.",
    "Balance throughput and stability. Use central_action keys only from the allowed policy list.",
    "Optimize congestion across NW NE SW SE. Return compact JSON only with at least one non-KEEP local action.",
    "Reduce hard-task queues while avoiding unstable phase switching. Include central_action.",
    "Prioritize corridor flow under congestion. Use bounded central policy deltas.",
    "Improve final_score by coordinating local phases and central traffic weights.",
] * 10

chat_prompts = [
    tokenizer.apply_chat_template(
        [
            {"role": "system", "content": "You output only compact valid JSON matching the required traffic schema."},
            {"role": "user", "content": SCHEMA_PROMPT + "\n" + user_prompt},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    for user_prompt in PROMPT_BANK_RAW
]

collected_data = []
GLOBAL_EPISODE = 0


def completion_to_text(completion):
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        return "\n".join(str(x.get("content", x)) if isinstance(x, dict) else str(x) for x in completion)
    return str(completion)


def safe_post(path, payload, retries=16, timeout=60):
    url = f"{ENV_URL}{path}"
    last_error = None
    for attempt in range(retries):
        try:
            response = requests.post(url, json=payload, timeout=timeout)
            if response.status_code in (429, 500, 502, 503, 504):
                wait = min(45, 2 * (attempt + 1)) + random.uniform(0.0, 1.0)
                print(f"HTTP {response.status_code} {path}; waiting {wait:.1f}s ({attempt + 1}/{retries})")
                time.sleep(wait)
                continue
            response.raise_for_status()
            return response
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError, requests.HTTPError) as exc:
            last_error = exc
            if isinstance(exc, requests.HTTPError) and getattr(exc.response, "status_code", None) == 422:
                raise
            wait = min(30, 2 + attempt)
            print(f"Request attempt {attempt + 1}/{retries} failed: {exc!r}; retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"Failed after {retries} retries: {url}. Last error: {last_error!r}")


def parse_action(completion):
    text = completion_to_text(completion)
    obj = extract_schema_json(text)
    if obj is not None:
        return obj, False, text
    return {"local_actions": {"NW": "KEEP", "NE": "KEEP", "SW": "KEEP", "SE": "KEEP"}, "central_action": {}}, True, text


def record_episode(data):
    collected_data.append(data)
    if use_wandb:
        flat = {k: v for k, v in data.items() if not isinstance(v, dict) and v is not None}
        for key, value in data.get("reward_breakdown", {}).items():
            if isinstance(value, (int, float)):
                flat[f"reward_breakdown/{key}"] = value
        try:
            wandb.log({f"episode/{k}": v for k, v in flat.items() if isinstance(v, (int, float, str))})
        except Exception as exc:
            print("W&B episode log skipped:", repr(exc))


def reward_fn(prompts, completions):
    global GLOBAL_EPISODE
    rewards = []
    for _, completion in zip(prompts, completions):
        GLOBAL_EPISODE += 1
        episode = GLOBAL_EPISODE
        task_id = "medium_dynamic" if episode < int(os.getenv("CURRICULUM_EPISODES", "40")) else "hard_multi"
        action, is_hallucinated, raw_text = parse_action(completion)

        if is_hallucinated:
            reward = -6.0
            print(f"[DEBUG] ep={episode} reward={reward:.4f} hallucinated=True raw={raw_text[:180]!r}")
            record_episode({"episode": episode, "task_id": task_id, "episode_reward": reward, "final_score": None, "reward_breakdown": {}, "step_count": 0, "is_hallucinated": 1.0, "is_all_keep": 0.0, "is_valid_action": 0.0, "has_central_action": 0.0, "raw_completion": raw_text[:500]})
            rewards.append(reward)
            continue

        if all(v == "KEEP" for v in action["local_actions"].values()):
            reward = -3.0
            print(f"[DEBUG] ep={episode} reward={reward:.4f} all_keep=True action={action}")
            record_episode({"episode": episode, "task_id": task_id, "episode_reward": reward, "final_score": None, "reward_breakdown": {}, "step_count": 0, "is_hallucinated": 0.0, "is_all_keep": 1.0, "is_valid_action": 0.0, "has_central_action": float(bool(action.get("central_action"))), "raw_completion": raw_text[:500]})
            rewards.append(reward)
            continue

        safe_post("/reset", {"task_id": task_id, "central_enabled": True})
        episode_reward = 0.0
        done = False
        step_count = 0
        info = {}
        latency_ms = 0.0
        while not done and step_count < int(os.getenv("MAX_ENV_STEPS", "30")):
            t0 = time.time()
            try:
                step_res = safe_post("/step", action)
            except requests.HTTPError as exc:
                if getattr(exc.response, "status_code", None) == 422:
                    episode_reward = -5.0
                    print(f"[DEBUG] ep={episode} reward={episode_reward:.4f} http422=True action={action}")
                    break
                raise
            latency_ms = (time.time() - t0) * 1000
            data = step_res.json()
            info = data.get("info", {}) or {}
            episode_reward += float(data.get("reward", 0.0))
            done = bool(data.get("done", False))
            step_count += 1
            time.sleep(float(os.getenv("ENV_STEP_SLEEP", "0.03")))

        if action.get("central_action"):
            episode_reward += 0.15
        else:
            episode_reward -= 0.25

        summary = info.get("summary", {}) if isinstance(info.get("summary", {}), dict) else {}
        reward_breakdown = info.get("reward_breakdown", {}) if isinstance(info.get("reward_breakdown", {}), dict) else {}
        final_score = summary.get("final_score", info.get("final_score", info.get("score", None)))
        print(f"[DEBUG] ep={episode} reward={episode_reward:.4f} score={final_score} task={task_id} central={bool(action.get('central_action'))} action={action}")
        record_episode({
            "episode": episode,
            "task_id": task_id,
            "episode_reward": float(episode_reward),
            "mean_queue": summary.get("mean_queue", info.get("mean_queue", None)),
            "mean_wait": summary.get("mean_wait", info.get("avg_wait", info.get("mean_wait", None))),
            "final_score": final_score,
            "throughput": summary.get("throughput", info.get("throughput", info.get("episode_throughput", None))),
            "throughput_efficiency": summary.get("throughput_efficiency", info.get("throughput_efficiency", None)),
            "reward_breakdown": reward_breakdown,
            "step_count": summary.get("step_count", step_count),
            "latency_ms": latency_ms,
            "is_hallucinated": 0.0,
            "is_all_keep": 0.0,
            "is_valid_action": 1.0,
            "has_central_action": float(bool(action.get("central_action"))),
            "action": action,
            "raw_completion": raw_text[:500],
        })
        rewards.append(float(episode_reward))
    return rewards

print("Reward function and prompts ready:", len(chat_prompts))

In [ ]:
# Cell 7: Manual GRPO-style policy optimization

optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=float(os.getenv("POLICY_LR", "5e-6")))
num_updates = int(os.getenv("POLICY_UPDATES", "80"))
prompts_per_update = int(os.getenv("PROMPTS_PER_UPDATE", "2"))
generations_per_prompt = int(os.getenv("GENERATIONS_PER_PROMPT", "4"))
max_new_tokens = int(os.getenv("MAX_NEW_TOKENS", "96"))
grad_clip = float(os.getenv("GRAD_CLIP", "1.0"))


def generate_completion(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=float(os.getenv("GEN_TEMPERATURE", "0.7")),
            top_p=float(os.getenv("GEN_TOP_P", "0.9")),
            pad_token_id=tokenizer.eos_token_id,
        )
    gen_ids = out[0, prompt_len:]
    completion = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return out[0].detach(), prompt_len, completion


def sequence_nll(full_ids, prompt_len):
    full_ids = full_ids.unsqueeze(0).to(model.device)
    attention_mask = torch.ones_like(full_ids, device=model.device)
    labels = full_ids.clone()
    labels[:, :prompt_len] = -100
    with torch.autocast("cuda", dtype=DTYPE):
        out = model(input_ids=full_ids, attention_mask=attention_mask, labels=labels)
        return out.loss

model.train()
model.config.use_cache = False
print("Starting manual GRPO-style training...")
print("updates:", num_updates, "episodes target:", num_updates * prompts_per_update * generations_per_prompt)

for update in range(1, num_updates + 1):
    batch_prompts = random.sample(chat_prompts, prompts_per_update)
    full_ids_list, prompt_lens, completions, prompt_refs, group_sizes = [], [], [], [], []
    for prompt in batch_prompts:
        start = len(completions)
        for _ in range(generations_per_prompt):
            full_ids, prompt_len, completion = generate_completion(prompt)
            full_ids_list.append(full_ids)
            prompt_lens.append(prompt_len)
            completions.append(completion)
            prompt_refs.append(prompt)
        group_sizes.append(len(completions) - start)

    rewards = reward_fn(prompt_refs, completions)
    advantages = []
    idx = 0
    for group_size in group_sizes:
        group_rewards = rewards[idx:idx + group_size]
        mean_r = sum(group_rewards) / len(group_rewards)
        std_r = (sum((r - mean_r) ** 2 for r in group_rewards) / len(group_rewards)) ** 0.5
        std_r = max(std_r, 1e-6)
        advantages.extend([(r - mean_r) / std_r for r in group_rewards])
        idx += group_size

    optimizer.zero_grad(set_to_none=True)
    losses = []
    for full_ids, prompt_len, adv in zip(full_ids_list, prompt_lens, advantages):
        nll = sequence_nll(full_ids, prompt_len)
        loss = nll * float(adv)
        loss.backward()
        losses.append(float(loss.detach().cpu()))
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    recent = collected_data[-len(rewards):]
    mean_reward = sum(rewards) / len(rewards)
    valid_rate = sum(1 for d in recent if d.get("is_valid_action") == 1.0) / len(rewards)
    central_rate = sum(1 for d in recent if d.get("has_central_action") == 1.0) / len(rewards)
    halluc_rate = sum(1 for d in recent if d.get("is_hallucinated") == 1.0) / len(rewards)
    scores = [d.get("final_score") for d in recent if d.get("final_score") is not None]
    mean_score = sum(scores) / len(scores) if scores else float("nan")
    best_score = max(scores) if scores else float("nan")
    mean_loss = sum(losses) / len(losses)

    print(f"[UPDATE {update:03d}] mean_reward={mean_reward:.4f} mean_score={mean_score:.5f} best_score={best_score:.5f} valid={valid_rate:.2f} central={central_rate:.2f} halluc={halluc_rate:.2f} loss={mean_loss:.4f}")
    if use_wandb:
        wandb.log({
            "manual_grpo/update": update,
            "manual_grpo/mean_reward": mean_reward,
            "manual_grpo/mean_final_score": mean_score,
            "manual_grpo/best_final_score": best_score,
            "manual_grpo/valid_rate": valid_rate,
            "manual_grpo/central_rate": central_rate,
            "manual_grpo/hallucination_rate": halluc_rate,
            "manual_grpo/loss": mean_loss,
        })

    if update % int(os.getenv("SAVE_EVERY_UPDATES", "20")) == 0:
        ckpt_dir = f"outputs/traffic-lora-a100-central-policy/checkpoint-update-{update}"
        model.save_pretrained(ckpt_dir)
        tokenizer.save_pretrained(ckpt_dir)
        print("Saved checkpoint:", ckpt_dir)

    torch.cuda.empty_cache()
    gc.collect()

print("Manual GRPO-style training complete.")
print("episodes recorded:", len(collected_data))

In [ ]:
# Cell 8: Save metrics, plots, adapter, and optional HF/W&B artifacts

import matplotlib.pyplot as plt

RUN_TAG = os.getenv("RUN_TAG", "a100_manual_grpo_central_policy")
OUT_DIR = Path("outputs") / "traffic-lora-a100-central-policy"
RESULTS_DIR = Path("results")
PLOTS_DIR = Path("plots")
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

if not collected_data:
    raise RuntimeError("No collected_data found. Run training before saving artifacts.")

model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("Saved adapter:", OUT_DIR)

metrics_json = RESULTS_DIR / "training_metrics_a100_central_policy.json"
metrics_csv = RESULTS_DIR / "training_metrics_a100_central_policy.csv"
with metrics_json.open("w") as f:
    json.dump(collected_data, f, indent=2, default=str)

df = pd.json_normalize(collected_data)
df.to_csv(metrics_csv, index=False)
print("Saved metrics:", metrics_json, metrics_csv)

for col in ["episode_reward", "final_score", "is_valid_action", "is_hallucinated", "has_central_action", "is_all_keep"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

plt.figure(figsize=(12, 6))
plt.plot(df["episode"], df["episode_reward"], label="A100 central-policy reward", color="blue")
plt.axhline(0, color="black", linestyle="--", linewidth=1, label="Zero reward")
plt.title("A100 Central-Policy Manual GRPO Reward Curve")
plt.xlabel("Episode")
plt.ylabel("Episode Reward")
plt.grid(alpha=0.25)
plt.legend()
reward_plot = PLOTS_DIR / "a100_central_policy_reward_curve.png"
plt.savefig(reward_plot, dpi=160, bbox_inches="tight")
plt.show()

if "final_score" in df.columns:
    score_df = df.dropna(subset=["final_score"])
    if len(score_df):
        plt.figure(figsize=(12, 6))
        plt.plot(score_df["episode"], score_df["final_score"], label="Final score", color="green")
        plt.axhline(0.49, color="red", linestyle="--", linewidth=1, label="Hard baseline reference")
        plt.title("A100 Central-Policy Final Score Progression")
        plt.xlabel("Episode")
        plt.ylabel("Final Score")
        plt.grid(alpha=0.25)
        plt.legend()
        score_plot = PLOTS_DIR / "a100_central_policy_final_score_curve.png"
        plt.savefig(score_plot, dpi=160, bbox_inches="tight")
        plt.show()

quality_cols = [c for c in ["is_valid_action", "has_central_action", "is_hallucinated", "is_all_keep"] if c in df.columns]
if quality_cols:
    plt.figure(figsize=(12, 6))
    for col in quality_cols:
        plt.plot(df["episode"], df[col].rolling(20, min_periods=1).mean(), label=col)
    plt.title("A100 Central-Policy Output Quality")
    plt.xlabel("Episode")
    plt.ylabel("Rolling Rate")
    plt.ylim(-0.05, 1.05)
    plt.grid(alpha=0.25)
    plt.legend()
    quality_plot = PLOTS_DIR / "a100_central_policy_output_quality.png"
    plt.savefig(quality_plot, dpi=160, bbox_inches="tight")
    plt.show()

summary = {
    "run_tag": RUN_TAG,
    "wandb_run_name": WANDB_RUN_NAME,
    "episodes": int(len(df)),
    "mean_reward": float(df["episode_reward"].mean()),
    "last_50_mean_reward": float(df["episode_reward"].tail(50).mean()),
    "best_reward": float(df["episode_reward"].max()),
    "valid_rate": float(df["is_valid_action"].mean()) if "is_valid_action" in df else None,
    "central_rate": float(df["has_central_action"].mean()) if "has_central_action" in df else None,
    "hallucination_rate": float(df["is_hallucinated"].mean()) if "is_hallucinated" in df else None,
    "mean_final_score": float(df["final_score"].dropna().mean()) if "final_score" in df else None,
    "best_final_score": float(df["final_score"].dropna().max()) if "final_score" in df else None,
}
summary_path = RESULTS_DIR / "summary_a100_central_policy.json"
with summary_path.open("w") as f:
    json.dump(summary, f, indent=2)
print("Summary:")
print(json.dumps(summary, indent=2))

if api:
    try:
        api.upload_folder(repo_id=SPACE_REPO, repo_type="space", folder_path=str(OUT_DIR), path_in_repo="outputs/traffic-lora-a100-central-policy", commit_message="Upload A100 central-policy adapter")
        api.upload_folder(repo_id=SPACE_REPO, repo_type="space", folder_path=str(RESULTS_DIR), path_in_repo="results", commit_message="Upload A100 central-policy metrics")
        api.upload_folder(repo_id=SPACE_REPO, repo_type="space", folder_path=str(PLOTS_DIR), path_in_repo="plots", commit_message="Upload A100 central-policy plots")
        print("Uploaded artifacts to HF Space:", SPACE_REPO)
    except Exception as exc:
        print("HF upload skipped/non-fatal:", repr(exc))
else:
    print("HF upload skipped because HF_TOKEN is missing.")

if use_wandb:
    try:
        artifact = wandb.Artifact(f"{WANDB_RUN_NAME}-artifacts", type="training-results")
        artifact.add_dir(str(OUT_DIR))
        artifact.add_file(str(metrics_json))
        artifact.add_file(str(metrics_csv))
        artifact.add_file(str(summary_path))
        for p in PLOTS_DIR.glob("a100_central_policy_*.png"):
            artifact.add_file(str(p))
        wandb.log_artifact(artifact)
        wandb.finish()
        print("Logged W&B artifact and finished run.")
    except Exception as exc:
        print("W&B artifact/finish skipped:", repr(exc))

In [ ]:
# Stop here for normal Run All execution.
# Legacy cells below are preserved only for historical reproducibility.

if os.getenv("RUN_LEGACY_EXPERIMENTS", "0") != "1":
    raise SystemExit("Final submission pipeline complete. Set RUN_LEGACY_EXPERIMENTS=1 only if you intentionally want to run legacy cells below.")

## Legacy Experimental Cells Below

The cells below are retained as historical context from earlier Kaggle/Colab iterations. The final submission path is Cells 1-8 above. Do not use the legacy Unsloth/TRL cells for the final run unless you are intentionally reproducing older experiments.

# Traffic Signal OpenEnv: Two-Stage Training Pipeline

Stage 1: SFT schema warmup teaches the model to emit valid traffic-action JSON.
Stage 2: GRPO uses the live OpenEnv traffic API only after schema generation passes validation.


In [ ]:
import os
import sys
import subprocess
import shutil
import glob
import site
import torch


def run_pip(args, *, required=True):
    print("pip", " ".join(args))
    result = subprocess.run([sys.executable, "-m", "pip", *args], check=False)
    if required and result.returncode != 0:
        raise RuntimeError(f"pip command failed: {' '.join(args)}")
    return result.returncode


print("Installing core dependencies without reinstalling torch...")
run_pip([
    "install", "--upgrade", "--no-cache-dir",
    "numpy", "requests", "matplotlib", "datasets", "wandb",
    "huggingface_hub", "accelerate", "peft", "bitsandbytes",
    "trl", "transformers",
])

cuda_tag = torch.__version__.split("+")[1] if "+" in torch.__version__ else ""
if cuda_tag.startswith("cu"):
    # Kaggle can keep a broken torchvision around. Clean leftovers first, then install a wheel matching torch/CUDA without changing torch.
    run_pip(["uninstall", "-y", "torchvision"], required=False)
    for base in site.getsitepackages():
        for path in glob.glob(os.path.join(base, "torchvision*")):
            print("Removing stale torchvision path:", path)
            if os.path.isdir(path):
                shutil.rmtree(path, ignore_errors=True)
            elif os.path.exists(path):
                os.remove(path)

    torchvision_version = None
    if torch.__version__.startswith("2.10."):
        torchvision_version = "torchvision==0.25.0"
    elif torch.__version__.startswith("2.9."):
        torchvision_version = "torchvision==0.24.0"
    else:
        torchvision_version = "torchvision"

    print(f"Installing {torchvision_version} for torch={torch.__version__}...")
    run_pip([
        "install", "--force-reinstall", "--no-cache-dir", "--no-deps",
        "--index-url", f"https://download.pytorch.org/whl/{cuda_tag}",
        torchvision_version,
    ], required=False)
else:
    print(f"Skipping torchvision install for non-CUDA torch build: {torch.__version__}")

print("Installing Unsloth...")
run_pip([
    "install", "--upgrade", "--no-cache-dir",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
])

print("Install complete. Restart kernel once, then rerun from the imports cell.")


In [ ]:
import os
import gc
import json
import re
import time
import random
import requests
import numpy as np
import torch

print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
try:
    import torchvision
    print("torchvision:", torchvision.__version__)
except Exception as exc:
    print("torchvision import warning (non-fatal unless Unsloth also fails):", repr(exc))

try:
    import wandb
except Exception as exc:
    wandb = None
    print("WandB import warning (training will continue without W&B):", repr(exc))

from datasets import Dataset
try:
    from unsloth import FastLanguageModel, PatchFastRL
except Exception as exc:
    raise RuntimeError(
        "Unsloth import failed. This is usually a torch/torchvision/CUDA mismatch. "
        "Run the install cell, restart kernel, then rerun imports. If Kaggle shows "
        "'operator torchvision::nms does not exist', rerun the install cell so it force-installs "
        "the matching torchvision wheel. Original error: " + repr(exc)
    ) from exc

try:
    from trl import GRPOTrainer, GRPOConfig, SFTTrainer, SFTConfig
except Exception as exc:
    raise RuntimeError(
        "TRL import failed. Run the install cell, restart kernel, then rerun imports. "
        "Original error: " + repr(exc)
    ) from exc

print("Imports ready.")


In [ ]:
PatchFastRL("GRPO", FastLanguageModel)

ENV_URL = os.getenv("ENV_URL", "https://guuru-dev-traffic-signal-openenv-2.hf.space")
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
print("ENV_URL:", ENV_URL)

WANDB_API_KEY = os.getenv("WANDB_API_KEY", "")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "traffic-signal-openenv")
WANDB_RUN_NAME = os.getenv("WANDB_RUN_NAME", f"openenv-1b-sft-grpo-{int(time.time())}")
use_wandb = bool(wandb) and len(WANDB_API_KEY) >= 40
if not use_wandb:
    os.environ["WANDB_DISABLED"] = "true"
    os.environ["WANDB_MODE"] = "disabled"
    wandb = None
    print("WandB disabled (missing/invalid WANDB_API_KEY).")
else:
    try:
        os.environ.pop("WANDB_DISABLED", None)
        os.environ["WANDB_MODE"] = "online"
        wandb.login(key=WANDB_API_KEY, relogin=True)
        wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config={
                "model_name": MODEL_NAME,
                "env_url": ENV_URL,
                "pipeline": "sft_schema_warmup_then_grpo",
                "target_model_size": "1B",
                "seed": 42,
            },
        )
        print(f"WandB initialized: project={WANDB_PROJECT}, run={WANDB_RUN_NAME}")
    except Exception as e:
        os.environ["WANDB_DISABLED"] = "true"
        os.environ["WANDB_MODE"] = "disabled"
        wandb = None
        use_wandb = False
        print(f"WandB init failed; continuing with WandB disabled: {e}")

r = requests.get(f"{ENV_URL}/health", timeout=30)
assert r.status_code == 200, f"Space not healthy: {r.status_code} {r.text}"
print("Space status:", r.json()["status"])
print("Task count:", r.json()["task_count"])

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected. Enable GPU before training.")
print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f"Seeds set to {SEED}.")

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16
model_dtype = torch.bfloat16 if use_bf16 else None
print(f"Precision config -> bf16={use_bf16}, fp16={use_fp16}, model_dtype={model_dtype}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    dtype=model_dtype,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.generation_config.do_sample = True
model.generation_config.temperature = 0.7
model.generation_config.top_p = 0.9
print("Model loaded.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
# Stage 1: supervised schema warmup
VALID_ACTIONS = {"KEEP", "SWITCH", "PHASE_0", "PHASE_1", "PHASE_2", "PHASE_3"}
CENTRAL_POLICY_KEYS = {
    "switch_penalty",
    "queue_urgency_weight",
    "emergency_boost",
    "corridor_priority",
    "balance_penalty",
}
SCHEMA_PROMPT = (
    'Return exactly one JSON object with this schema: '
    '{"local_actions":{"NW":"ACTION","NE":"ACTION","SW":"ACTION","SE":"ACTION"},'
    '"central_action":{"queue_urgency_weight":0.0,"corridor_priority":0.0}}. '
    'ACTION must be one of KEEP, SWITCH, PHASE_0, PHASE_1, PHASE_2, PHASE_3. '
    'central_action is optional but may only use switch_penalty, queue_urgency_weight, emergency_boost, corridor_priority, balance_penalty. '
    'central_action values must be small numeric deltas from -0.5 to 0.8. No prose.'
)

schema_examples = [
    '{"local_actions":{"NW":"SWITCH","NE":"KEEP","SW":"PHASE_0","SE":"PHASE_1"},"central_action":{"queue_urgency_weight":0.4,"corridor_priority":0.3}}',
    '{"local_actions":{"NW":"PHASE_2","NE":"SWITCH","SW":"KEEP","SE":"PHASE_0"},"central_action":{"balance_penalty":0.3,"switch_penalty":0.2}}',
    '{"local_actions":{"NW":"PHASE_1","NE":"PHASE_3","SW":"SWITCH","SE":"KEEP"},"central_action":{"emergency_boost":0.5,"queue_urgency_weight":0.2}}',
    '{"local_actions":{"NW":"PHASE_0","NE":"PHASE_1","SW":"PHASE_2","SE":"SWITCH"},"central_action":{"corridor_priority":0.5,"balance_penalty":0.2}}',
    '{"local_actions":{"NW":"SWITCH","NE":"PHASE_0","SW":"PHASE_1","SE":"PHASE_2"},"central_action":{"switch_penalty":0.3,"queue_urgency_weight":0.3}}',
    '{"local_actions":{"NW":"PHASE_3","NE":"PHASE_2","SW":"SWITCH","SE":"PHASE_0"},"central_action":{"emergency_boost":0.4,"balance_penalty":0.3}}',
]

rows = []
for _ in range(120):
    for example in schema_examples:
        text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": "You output only compact valid JSON matching the required traffic schema."},
                {"role": "user", "content": SCHEMA_PROMPT},
                {"role": "assistant", "content": example},
            ],
            tokenize=False,
            add_generation_prompt=False,
        )
        rows.append({"text": text})

sft_dataset = Dataset.from_list(rows)

sft_args = SFTConfig(
    output_dir="./outputs-sft-schema",
    max_steps=80,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    max_seq_length=512,
    packing=False,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="none",
    logging_steps=1,
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset,
    args=sft_args,
    dataset_text_field="text",
)

sft_trainer.train()
print("Schema SFT warmup complete.")


In [ ]:
# Validate schema generation before starting GRPO
LOCAL_KEY_ALIASES = {"localactions", "localaction", "local", "actions", "action"}
CENTRAL_KEY_ALIASES = {"centralaction", "centralactions", "central", "policy"}


def _alias_key(key):
    return re.sub(r"[^a-z0-9]", "", str(key).lower())


def normalize_action_obj(obj):
    if not isinstance(obj, dict):
        return None

    local = None
    central = {}
    for key, value in obj.items():
        alias = _alias_key(key)
        if alias in LOCAL_KEY_ALIASES:
            local = value
        elif alias in CENTRAL_KEY_ALIASES:
            central = value

    if not isinstance(local, dict):
        return None

    clean = {}
    for direction in ("NW", "NE", "SW", "SE"):
        raw_value = None
        for key, value in local.items():
            if str(key).upper().strip() == direction:
                raw_value = value
                break
        if raw_value is None:
            return None
        action = str(raw_value).upper().strip()
        if action not in VALID_ACTIONS:
            return None
        clean[direction] = action

    clean_central = {}
    if isinstance(central, dict):
        for key, value in central.items():
            if key not in CENTRAL_POLICY_KEYS:
                return None
            try:
                clean_central[key] = max(-0.5, min(0.8, float(value)))
            except Exception:
                return None
    elif central not in ({}, None):
        return None

    return {"local_actions": clean, "central_action": clean_central}


def extract_schema_json(text):
    decoder = json.JSONDecoder()
    best = None
    for idx, char in enumerate(text):
        if char != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(text[idx:])
        except Exception:
            continue
        normalized = normalize_action_obj(obj)
        if normalized is not None:
            best = normalized
    return best


def is_valid_schema(obj):
    return normalize_action_obj(obj) is not None


valid_count = 0
for i in range(5):
    messages = [
        {"role": "system", "content": "You output only compact valid JSON matching the required traffic schema."},
        {"role": "user", "content": SCHEMA_PROMPT},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=96,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    obj = extract_schema_json(text)
    ok = is_valid_schema(obj)
    valid_count += int(ok)
    print(f"SAMPLE {i + 1} valid={ok} extracted={obj}")

SCHEMA_READY = valid_count >= 3
print(f"Schema validation: {valid_count}/5 valid")
if not SCHEMA_READY:
    raise RuntimeError("Schema warmup insufficient. Do not start GRPO; rerun SFT warmup or increase max_steps.")


In [ ]:
# Stage 2: GRPO environment fine-tuning, gated on schema readiness
if not globals().get("SCHEMA_READY", False):
    raise RuntimeError("SCHEMA_READY is false. Run schema validation before GRPO.")

PROMPT_BANK_RAW = [
    'Hard multi-intersection control: choose local phase actions and small central deltas for queue pressure and corridor flow.',
    'Prevent spillback using valid local actions plus bounded central_action deltas. JSON only.',
    'Emergency-aware traffic control: include emergency_boost only if useful, keep deltas small.',
    'Balance throughput and stability. Use central_action keys only from the allowed policy list.',
    'Optimize congestion across NW NE SW SE. Return compact JSON only with at least one non-KEEP local action.',
] * 8

chat_prompts = []
for user_prompt in PROMPT_BANK_RAW:
    chat_prompts.append(
        tokenizer.apply_chat_template(
            [
                {"role": "system", "content": "You output only compact valid JSON matching the required traffic schema."},
                {"role": "user", "content": SCHEMA_PROMPT + "\n" + user_prompt},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
    )

train_dataset = Dataset.from_dict({"prompt": chat_prompts})
collected_data = []
GLOBAL_EPISODE = 0

def safe_post(url, payload, retries=16, timeout=60):
    for attempt in range(retries):
        try:
            rr = requests.post(url, json=payload, timeout=timeout)
            if rr.status_code in (429, 500, 502, 503, 504):
                wait = min(45, 2 * (attempt + 1)) + random.uniform(0.0, 1.0)
                print(f"HTTP {rr.status_code}. Waiting {wait:.1f}s ({attempt+1}/{retries})")
                time.sleep(wait)
                continue
            rr.raise_for_status()
            return rr
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            wait = min(30, 2 + attempt)
            print(f"Network/timeout attempt {attempt+1}/{retries}. Retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"Failed after {retries} retries: {url}")

def parse_action(completion):
    obj = extract_schema_json(completion)
    if is_valid_schema(obj):
        return obj, False
    return {"local_actions": {"NW": "KEEP", "NE": "KEEP", "SW": "KEEP", "SE": "KEEP"}, "central_action": {}}, True


def record_episode(data):
    collected_data.append(data)
    if use_wandb and wandb:
        flat = {k: v for k, v in data.items() if not isinstance(v, dict) and v is not None}
        for key, value in data.get("reward_breakdown", {}).items():
            flat[f"reward_breakdown/{key}"] = value
        wandb.log({f"episode/{k}": v for k, v in flat.items()})


def reward_fn(prompts, completions, **kwargs):
    global GLOBAL_EPISODE
    rewards = []
    for _, completion in zip(prompts, completions):
        GLOBAL_EPISODE += 1
        episode = GLOBAL_EPISODE
        task_id = "medium_dynamic" if episode < 20 else "hard_multi"
        action, is_hallucinated = parse_action(completion)

        if is_hallucinated:
            reward = -6.0
            print(f"[DEBUG] ep={episode} reward={reward:.4f} hallucinated=True raw={completion[:160]!r}")
            record_episode({
                "episode": episode,
                "task_id": task_id,
                "episode_reward": reward,
                "final_score": None,
                "mean_queue": None,
                "mean_wait": None,
                "throughput": None,
                "throughput_efficiency": None,
                "reward_breakdown": {},
                "step_count": 0,
                "is_hallucinated": 1.0,
                "is_all_keep": 0.0,
                "is_valid_action": 0.0,
                "raw_completion": completion[:500],
            })
            rewards.append(reward)
            continue

        if all(v == "KEEP" for v in action["local_actions"].values()):
            reward = -3.0
            print(f"[DEBUG] ep={episode} reward={reward:.4f} all_keep=True action={action}")
            record_episode({
                "episode": episode,
                "task_id": task_id,
                "episode_reward": reward,
                "final_score": None,
                "mean_queue": None,
                "mean_wait": None,
                "throughput": None,
                "throughput_efficiency": None,
                "reward_breakdown": {},
                "step_count": 0,
                "is_hallucinated": 0.0,
                "is_all_keep": 1.0,
                "is_valid_action": 0.0,
                "raw_completion": completion[:500],
            })
            rewards.append(reward)
            continue

        safe_post(f"{ENV_URL}/reset", {"task_id": task_id, "central_enabled": True})
        episode_reward = 0.0
        done = False
        step_count = 0
        info = {}
        latency_ms = 0.0

        while not done and step_count < 25:
            t0 = time.time()
            try:
                step_res = safe_post(f"{ENV_URL}/step", action)
            except requests.HTTPError as exc:
                if getattr(exc.response, "status_code", None) == 422:
                    episode_reward = -5.0
                    print(f"[DEBUG] ep={episode} reward={episode_reward:.4f} http422=True action={action}")
                    break
                raise
            latency_ms = (time.time() - t0) * 1000
            data = step_res.json()
            info = data.get("info", {})
            episode_reward += float(data.get("reward", 0.0))
            done = data.get("done", False)
            step_count += 1
            time.sleep(0.05)

        summary = info.get("summary", {}) if isinstance(info.get("summary", {}), dict) else {}
        reward_breakdown = info.get("reward_breakdown", {}) if isinstance(info.get("reward_breakdown", {}), dict) else {}
        final_score = summary.get("final_score", info.get("final_score", info.get("score", 0.0)))

        print(f"[DEBUG] ep={episode} reward={episode_reward:.4f} score={final_score:.4f} task={task_id} valid=True action={action}")
        record_episode({
            "episode": episode,
            "task_id": task_id,
            "episode_reward": episode_reward,
            "mean_queue": summary.get("mean_queue", info.get("mean_queue", 0.0)),
            "mean_wait": summary.get("mean_wait", info.get("avg_wait", info.get("mean_wait", 0.0))),
            "final_score": final_score,
            "throughput": summary.get("throughput", info.get("throughput", info.get("episode_throughput", 0.0))),
            "throughput_efficiency": summary.get("throughput_efficiency", info.get("throughput_efficiency", 0.0)),
            "reward_breakdown": reward_breakdown,
            "step_count": summary.get("step_count", step_count),
            "is_hallucinated": 0.0,
            "is_all_keep": 0.0,
            "is_valid_action": 1.0,
            "action": action,
        })
        rewards.append(episode_reward)
    return rewards

grpo_args = GRPOConfig(
    output_dir="./outputs",
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    max_steps=80,
    max_prompt_length=512,
    max_completion_length=96,
    num_generations=4,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="wandb" if use_wandb else "none",
    run_name=WANDB_RUN_NAME if use_wandb else None,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    logging_steps=1,
)

grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=grpo_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
if not use_wandb:
    grpo_trainer.args.report_to = []
    grpo_trainer.callback_handler.callbacks = [cb for cb in grpo_trainer.callback_handler.callbacks if "wandb" not in cb.__class__.__name__.lower()]

print("Starting GRPO environment training...")
torch.cuda.empty_cache()
gc.collect()
grpo_trainer.train()

model.save_pretrained("outputs/traffic-lora")
tokenizer.save_pretrained("outputs/traffic-lora")
print("Adapter weights saved to outputs/traffic-lora")


In [ ]:
print("WandB run still open for artifact upload." if use_wandb and wandb else "WandB disabled for this run.")


In [ ]:
# Auto-save adapter, metrics, and plots to HF Hub
import os
import sys
import json
import csv
from huggingface_hub import HfApi

HF_TOKEN = os.getenv("HF_TOKEN")
api = HfApi(token=HF_TOKEN) if HF_TOKEN else None
if not HF_TOKEN:
    print("HF_TOKEN not set; local metrics/plots will be saved, but Hub upload will be skipped.")

if not os.path.exists("outputs/traffic-lora"):
    raise RuntimeError("outputs/traffic-lora not found. Training may not have completed.")
print("Model weights found.")
print("Contents:", os.listdir("outputs/traffic-lora"))

if api:
    try:
        api.upload_folder(
            folder_path="outputs/traffic-lora",
            repo_id="Guuru-DEV/traffic-signal-openenv-2",
            repo_type="space",
            path_in_repo="outputs/traffic-lora",
            token=HF_TOKEN,
        )
        print("Adapter uploaded to HF Space: https://huggingface.co/spaces/Guuru-DEV/traffic-signal-openenv-2")
    except Exception as e:
        print(f"Adapter upload skipped (non-fatal): {e}")

if "collected_data" not in globals():
    raise RuntimeError("collected_data not found. Ensure GRPO cell completed.")

os.makedirs("results", exist_ok=True)
os.makedirs("plots", exist_ok=True)
metrics_json_path = "results/training_metrics.json"
metrics_csv_path = "results/training_metrics.csv"

with open(metrics_json_path, "w") as f:
    json.dump(collected_data, f, indent=2)

flat_rows = []
for row in collected_data:
    flat = {k: v for k, v in row.items() if not isinstance(v, dict)}
    for key, value in row.get("reward_breakdown", {}).items():
        flat[f"reward_breakdown_{key}"] = value
    flat_rows.append(flat)

fieldnames = sorted({key for row in flat_rows for key in row.keys()})
with open(metrics_csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(flat_rows)
print(f"Saved metrics to {metrics_json_path} and {metrics_csv_path}")

try:
    sys.path.insert(0, ".")
    from env.metrics_exporter import generate_training_plots
    generate_training_plots(collected_data, "plots")
    print("Plots generated in plots/")
except Exception as e:
    print(f"Plot generation skipped (non-fatal): {e}")

if api:
    try:
        api.upload_folder(
            folder_path="results",
            repo_id="Guuru-DEV/traffic-signal-openenv-2",
            repo_type="space",
            path_in_repo="results",
            token=HF_TOKEN,
        )
        print("Metrics uploaded to HF Space: https://huggingface.co/spaces/Guuru-DEV/traffic-signal-openenv-2/tree/main/results")
    except Exception as e:
        print(f"Metrics upload skipped (non-fatal): {e}")

if os.path.exists("plots") and os.listdir("plots"):
    if api:
        try:
            api.upload_folder(
                folder_path="plots",
                repo_id="Guuru-DEV/traffic-signal-openenv-2",
                repo_type="space",
                path_in_repo="plots",
                token=HF_TOKEN,
            )
            print("Plots uploaded to HF Space: https://huggingface.co/spaces/Guuru-DEV/traffic-signal-openenv-2/tree/main/plots")
        except Exception as e:
            print(f"Plot upload skipped (non-fatal): {e}")
else:
    print("No plots found; skipping plot upload.")

if use_wandb and wandb:
    artifact = wandb.Artifact(f"{WANDB_RUN_NAME}-results", type="training-results")
    artifact.add_dir("results")
    if os.path.exists("plots") and os.listdir("plots"):
        artifact.add_dir("plots")
    wandb.log_artifact(artifact)
    wandb.finish()
    print("WandB artifacts logged and run finished.")

print("Training artifact save step complete.")
